In [1]:
import sys
sys.path.append("..")

import numpy as np
from src.preprocess import preprocess
from src.bm25 import BM25
from src.eval import evaluate
from src.semantic import SemanticSearch

d:\search_keyword_semantic\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from beir.datasets.data_loader import GenericDataLoader

corpus, queries, qrels = GenericDataLoader("../data/scifact").load(split="test")
docs = {did: (d["title"] + ". " + d["text"]).strip() for did, d in corpus.items()}
doc_ids = list(docs)
texts = [docs[d] for d in doc_ids]
print(len(texts), len(queries))

100%|██████████| 5183/5183 [00:00<00:00, 133092.59it/s]

5183 300


In [3]:
from sentence_transformers import SentenceTransformer

demo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
s = ["Vitamin B9 deficiency raises blood homocysteine",
     "A lack of folate increases homocysteine levels",
     "Tau phosphorylation in neurons"]
v = demo.encode(s, normalize_embeddings=True)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("sentence 1 vs 2 (same meaning, different words):", round(float(cosine(v[0], v[1])), 3))
print("sentence 1 vs 3 (unrelated):                    ", round(float(cosine(v[0], v[2])), 3))
print("Words shared after preprocessing:", set(preprocess(s[0])) & set(preprocess(s[1])))

d:\search_keyword_semantic\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\swapn\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\search_keyword_semantic\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarnin

sentence 1 vs 2 (same meaning, different words): 0.704
sentence 1 vs 3 (unrelated):                     0.029
Words shared after preprocessing: {'homocystein'}


In [4]:
import os

semantic = SemanticSearch(doc_ids)

emb_path = "../data/scifact_minilm_embeddings.npy"
if os.path.exists(emb_path):
    semantic.load(emb_path)
else:
    semantic.build(texts)             # a few minutes on CPU, with a progress bar
    semantic.save(emb_path)

print(semantic.doc_emb.shape)         # expect (5183, 384)

Batches: 100%|██████████| 81/81 [03:39<00:00,  2.71s/it]

(5183, 384)


In [5]:
qid = list(queries)[0]
print("QUERY:", queries[qid])
relevant = set(qrels[qid])
for rank, (did, s) in enumerate(semantic.search(queries[qid], k=5), 1):
    mark = "✔" if did in relevant else " "
    print(f"{rank}. {mark} {s:.3f} | {docs[did][:90]}")

QUERY: 0-dimensional biomaterials show inductive properties.
1.   0.359 | Complex Tissue and Disease Modeling using hiPSCs.. Defined genetic models based on human p
2.   0.334 | Nonlinear Elasticity in Biological Gels. Unlike most synthetic materials, biological mater
3.   0.295 | An efficient nonviral method to generate integration-free human-induced pluripotent stem c
4.   0.293 | Induced pluripotent stem cell lines derived from human somatic cells.. Somatic cell nuclea
5. ✔ 0.291 | New opportunities: the use of nanotechnologies to manipulate and track stem cells.. Nanote


In [6]:
semantic_results = semantic.search_many(queries, k=10)
print("Semantic:", evaluate(semantic_results, qrels))

Semantic: {'Recall@10': 0.7883, 'MRR@10': 0.6068, 'nDCG@10': 0.6484}


In [7]:
tokenized = [preprocess(t) for t in texts]
bm25 = BM25(tokenized, doc_ids)

bm25_results = {qid: [d for d, _ in bm25.search(preprocess(q), k=10)]
                for qid, q in queries.items()}
print("BM25    :", evaluate(bm25_results, qrels))

BM25    : {'Recall@10': 0.8302, 'MRR@10': 0.6527, 'nDCG@10': 0.6903}


In [8]:
import pandas as pd, os

table = pd.DataFrame({
    "BM25":     evaluate(bm25_results, qrels),
    "Semantic": evaluate(semantic_results, qrels),
}).T
print(table)

os.makedirs("../results", exist_ok=True)
table.to_csv("../results/day3_bm25_vs_semantic.csv")

          Recall@10  MRR@10  nDCG@10
BM25         0.8302  0.6527   0.6903
Semantic     0.7883  0.6068   0.6484


In [9]:
def first_hit(results, qid):
    for i, d in enumerate(results[qid]):
        if d in qrels[qid]:
            return i + 1
    return None

bm25_only, sem_only, both_miss = [], [], []
for qid in queries:
    b, s = first_hit(bm25_results, qid), first_hit(semantic_results, qid)
    if b and not s:
        bm25_only.append(qid)
    elif s and not b:
        sem_only.append(qid)
    elif not b and not s:
        both_miss.append(qid)

print("Only BM25 found it:    ", len(bm25_only))
print("Only semantic found it:", len(sem_only))
print("Both missed it:        ", len(both_miss))

Only BM25 found it:     28
Only semantic found it: 13
Both missed it:         32


In [10]:
def show(qid):
    print("QUERY:", queries[qid])
    for did in qrels[qid]:
        print("RELEVANT:", docs[did][:300], "\n")

if sem_only:
    print("--- Semantic found, BM25 missed ---"); show(sem_only[0])
if bm25_only:
    print("--- BM25 found, semantic missed ---"); show(bm25_only[0])

--- Semantic found, BM25 missed ---
QUERY: 0-dimensional biomaterials show inductive properties.
RELEVANT: New opportunities: the use of nanotechnologies to manipulate and track stem cells.. Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo track 

--- BM25 found, semantic missed ---
QUERY: A total of 1,000 people in the UK are asymptomatic carriers of vCJD infection.
RELEVANT: Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey. OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection  

